# CRIVO — Enciclopédia: o começo dos verbetes da Wikipédia em português

Monta a enciclopédia de consulta do CRIVO (`scripts/construir_enciclopedia.py`). São a primeira ou as duas primeiras frases dos **120 mil verbetes mais longos** da Wikipédia em português (`wikimedia/wikipedia`, 20231101.pt, as 6 partes, revisão fixa, licença CC BY-SA). Quando o acervo dele não sabe um assunto, ele procura o verbete pelo título e diz essa frase, com a fonte.

Não é modelo nem texto gerado por modelo: é texto escrito por pessoas, como a Wikipédia que ele já leu no pré-treino.

**Não precisa de GPU.** Ambiente de execução → Executar tudo (uns 10 a 15 minutos; baixa ~1,8 GB). Resultado em `CRIVO/enciclopedia-pt/`, em partes de 3,5 MB, para o assistente pegar.

In [ ]:
# Célula 0 — Drive e código do CRIVO
import os, sys, json, shutil, subprocess, hashlib, time, urllib.request
from pathlib import Path
from google.colab import drive
drive.mount("/content/drive", force_remount=False)
MEU = Path("/content/drive/MyDrive")
candidatas = [MEU / "CRIVO"] + sorted(p for p in MEU.iterdir() if p.is_dir() and p.name != "CRIVO")
CRIVO = next((p for p in candidatas if (p / "dados-17m").is_dir()), None)
if CRIVO is None:
    raise SystemExit("ERRO: não achei a pasta CRIVO no Drive desta conta.")
REPO = Path("/content/CRIVO")
shutil.rmtree(REPO, ignore_errors=True)
subprocess.run(["git", "clone", "--depth", "1", "--branch", "claude/affectionate-carson-22yoe0",
                "https://github.com/HROSONE/CRIVO", str(REPO)], check=True)
sys.path.insert(0, str(REPO / "scripts")); sys.path.insert(0, str(REPO))
import construir_enciclopedia as C
LOCAL = Path("/content/wiki")
LOCAL.mkdir(exist_ok=True)
def sha(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""):
            h.update(b)
    return h.hexdigest()
print("CRIVO:", CRIVO, flush=True)


In [ ]:
# Célula 1 — as 6 partes da Wikipédia em português (revisão fixa; o SHA de cada uma fica no manifesto)
REVISAO = "b04c8d1ceb2f5cd4588862100d08de323dccfbaa"
PARTES = ["20231101.pt/train-%05d-of-00006.parquet" % i for i in range(6)]
baixadas = []
for nome in PARTES:
    destino = LOCAL / nome.split("/")[-1]
    if not destino.is_file():
        url = f"https://huggingface.co/datasets/wikimedia/wikipedia/resolve/{REVISAO}/{nome}"
        t0 = time.time()
        with urllib.request.urlopen(url, timeout=300) as r, open(destino.with_suffix(".tmp"), "wb") as f:
            shutil.copyfileobj(r, f, 8 << 20)
        destino.with_suffix(".tmp").replace(destino)
        print(f"  {nome}: {destino.stat().st_size / 1e6:.0f} MB em {time.time() - t0:.0f}s", flush=True)
    baixadas.append({"arquivo": nome, "sha256": sha(destino), "bytes": destino.stat().st_size})
print("partes:", len(baixadas), flush=True)


In [ ]:
# Célula 2 — construir (120 mil verbetes) e conferir alguns
t0 = time.time()
ENC = C.construir(C.ler_parquets([LOCAL / b["arquivo"].split("/")[-1] for b in baixadas]), n=120_000)
SAIDA_LOCAL = LOCAL / "saida"
GZ_SHA = C.salvar(ENC, SAIDA_LOCAL)
GZ = SAIDA_LOCAL / "enciclopedia_pt.json.gz"
print(f"{ENC['n']} verbetes, {len(ENC['chaves'])} chaves, {GZ.stat().st_size / 1e6:.1f} MB, sha {GZ_SHA} ({time.time() - t0:.0f}s)", flush=True)
import enciclopedia
enciclopedia.ARQUIVO = GZ
enciclopedia._dados.cache_clear()
for fala in ["o que é a Mongólia?", "o que é um ornitorrinco?", "quem foi Pelé?", "me fala sobre a Revolução Russa",
             "o que é inflação?", "quem foi Carlos Drummond de Andrade?", "o que é um buraco negro?", "o que é o Pantanal?",
             "quem foi Marie Curie?", "o que é DNA?", "o que é a Lua?", "o que é capoeira?"]:
    achado = enciclopedia.consultar(fala)
    print(f"  {fala!r} → {achado}", flush=True)


In [ ]:
# Célula 3 — partes de até 3,5 MB no Drive
SAIDA = CRIVO / "enciclopedia-pt"
SAIDA.mkdir(parents=True, exist_ok=True)
for antigo in SAIDA.glob("parte_*"):
    antigo.unlink()
dados = GZ.read_bytes()
TAM = 3584 * 1024
partes = []
for i in range(0, len(dados), TAM):
    nome = "parte_%02d.bin" % (i // TAM)
    (SAIDA / nome).write_bytes(dados[i:i + TAM])
    partes.append({"nome": nome, "sha256": hashlib.sha256(dados[i:i + TAM]).hexdigest()})
manifesto = {"gz_sha256": GZ_SHA, "gz_bytes": len(dados), "verbetes": ENC["n"], "chaves": len(ENC["chaves"]),
             "fonte": ENC["fonte"], "licenca": ENC["licenca"], "revisao": REVISAO, "partes_wikipedia": baixadas, "partes": partes}
(SAIDA / "manifesto_enciclopedia.json").write_text(json.dumps(manifesto, ensure_ascii=False, indent=1))
print(json.dumps({k: v for k, v in manifesto.items() if k not in ("partes", "partes_wikipedia")}, ensure_ascii=False, indent=1), "\n", len(partes), "partes")
print("\nPRONTO: avise o assistente.", flush=True)
